# 🎬 Movie Recommendation System

## Project Objective

This project recommends movies similar to a user's selected movie using Content-Based Filtering.

In [2]:
import pandas as pd
import numpy as np
import ast

# Step 2: Load the Dataset

The TMDB Movies and Credits datasets are loaded into Pandas DataFrames for further analysis.

In [3]:
movies=pd.read_csv("tmdb_5000_movies.csv")
credits=pd.read_csv("tmdb_5000_credits.csv")
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [4]:
credits.head() # first five rows

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [5]:
movies.shape 

(4803, 20)

In [6]:
credits.shape

(4803, 4)

# Step 3: Merge the Datasets

Both datasets are merged using the movie title column to obtain complete movie information in a single DataFrame.


In [7]:
movies=movies.merge(credits,on='title')
movies.shape

(4809, 23)

# Step 4: Select Important Features

Only the columns relevant for building the recommendation system are selected.

Selected Features:
- Movie ID
- Title
- Overview
- Genres
- Keywords
- Cast
- Crew

In [8]:
movies=movies[
    ['movie_id',
     'title',
     'keywords',
     'overview',
     'genres',
     'cast',
     'crew'
    ]
    ]
movies.head()

,movie_id,title,keywords,overview,genres,cast,crew
0,19995,Avatar,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


# Step 5: Handle Missing Values

Missing values can affect recommendation quality. Therefore, null values are identified and removed.

In [9]:
print(movies.isnull().sum())

movie_id    0
title       0
keywords    0
overview    3
genres      0
cast        0
crew        0
dtype: int64


In [10]:
movies.dropna(inplace=True)
print(movies.isnull().sum())

movie_id    0
title       0
keywords    0
overview    0
genres      0
cast        0
crew        0
dtype: int64


In [11]:
print(movies.duplicated().sum())

0


In [12]:
movies['genres'].iloc[0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

# Step 6: Extract Genre Information and Keywords

The genres column and keyword is stored in JSON-like format. We extract only the genre names and keywords for each movie.

In [13]:
import ast

def convert(text):

    if isinstance(text, list):
        return text

    L=[]

    for i in ast.literal_eval(text):
        L.append(i['name'])

    return L

In [14]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

# Step 7: Extract Top Cast Members

Only the top three cast members are extracted because lead actors significantly influence movie similarity.

In [15]:
import ast

def convert_cast(text):

    L = []
    counter = 0

    for i in ast.literal_eval(text):

        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

In [16]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [17]:
movies['cast'].iloc[0]

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']

# Step 8: Extract Director Information

The director's name is extracted from the crew column because directors often define a movie's style and storytelling approach.

In [18]:
def fetch_director(text):

    L = []

    for i in ast.literal_eval(text):

        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

In [19]:
movies['crew'] = movies['crew'].apply(fetch_director)


In [20]:
movies['crew'].iloc[0]

['James Cameron']

# Step 9: Process Movie Overview

The overview text is tokenized by splitting sentences into individual words.

In [21]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies['overview'].iloc[0]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.']

# Step 10: Remove Spaces from Multi-Word Terms

Spaces are removed from names such as:

- Sam Worthington → SamWorthington
- Science Fiction → ScienceFiction

This ensures they are treated as single features.

In [22]:
movies['genres'] = movies['genres'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['keywords'] = movies['keywords'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['cast'] = movies['cast'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['crew'] = movies['crew'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

# Step 11: Create Tags Feature

All important movie information is combined into a single column called Tags.

Tags contain:
- Overview
- Genres
- Keywords
- Cast
- Director

This becomes the primary feature used for recommendations.

In [23]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies['tags'].iloc[0]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.',
 'Action',
 'Adventure',
 'Fantasy',
 'ScienceFiction',
 'cultureclash',
 'future',
 'spacewar',
 'spacecolony',
 'society',
 'spacetravel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alienplanet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'loveaffair',
 'antiwar',
 'powerrelations',
 'mindandsoul',
 '3d',
 'SamWorthington',
 'ZoeSaldana',
 'SigourneyWeaver',
 'JamesCameron']

# Step 12: Create Final Dataset

A new DataFrame is created containing only:

- Movie ID
- Movie Title
- Tags

In [24]:
new_df = movies[['movie_id','title','tags']]

# Step 13: Convert Tags into Text Format

The tags stored as lists are converted into a single string for text processing.

In [25]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

C:\Users\acer\AppData\Local\Temp\ipykernel_4220\1824047427.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


# Step 14: Text Normalization

All text is converted to lowercase to maintain consistency and avoid duplicate words caused by capitalization differences.

In [26]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\acer\AppData\Local\Temp\ipykernel_4220\1380776331.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [27]:
print(new_df.head())
print(type(new_df['tags'].iloc[0]))

   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   
2    206647                                   Spectre   
3     49026                     The Dark Knight Rises   
4     49529                               John Carter   

                                                tags  
0  in the 22nd century, a paraplegic marine is di...  
1  captain barbossa, long believed to be dead, ha...  
2  a cryptic message from bond’s past sends him o...  
3  following the death of district attorney harve...  
4  john carter is a war-weary, former military ca...  
<class 'str'>


# Step 15: Apply Stemming

Stemming reduces words to their root form.

Examples:
- Loved → Love
- Loving → Love
- Movies → Movie

This improves text matching and similarity detection.

In [28]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\acer\AppData\Local\Temp\ipykernel_4220\538954741.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


# Step 16: Vectorization using CountVectorizer

Machine learning algorithms cannot understand text directly.

CountVectorizer converts movie tags into numerical vectors that represent word frequencies.

In [29]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()

In [30]:
vectors.shape

(4806, 5000)

# Step 17: Calculate Cosine Similarity

Cosine Similarity measures the similarity between movie vectors.

A higher similarity score indicates that two movies share more common features.

In [31]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [32]:
similarity.shape

(4806, 4806)

# Step 18: Build the Recommendation Function

A recommendation function is created that:

1. Finds the selected movie.
2. Calculates similarity scores.
3. Sorts movies by similarity.
4. Returns the top 5 recommendations.

In [33]:
def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movie_list:
        print(new_df.iloc[i[0]].title)
        

# Step 19: Test the Recommendation System

The recommendation engine is tested using a sample movie title.

In [36]:
recommend('The Avengers')

Iron Man 3
Avengers: Age of Ultron
Captain America: Civil War
Captain America: The First Avenger
Iron Man


# Conclusion

The Movie Recommendation System was successfully developed using Content-Based Filtering.

Key Techniques Used:
- Data Preprocessing
- Feature Engineering
- Natural Language Processing
- Count Vectorization
- Cosine Similarity

The system effectively recommends movies with similar content and characteristics.